<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 7

### Details
Coinbase earnings call transcript:
https://www.fool.com/earnings/call-transcripts/2024/10/30/coinbase-global-coin-q3-2024
-earnings-call-transcr/

The Base Question: "Is Coinbase's strategy to reduce reliance on volatile trading fees
actually working as of Q3 2024?"
Use four different prompting techniques to answer the question. Use an LLM as a judge
to rate the results from different prompts. Based on the "Judge's" feedback, write a final
200-word Executive Summary answering the Base Question.

---


### What I found interesting
- Meta-layer: LLM assessing the quality of other LLMs
- Complexity awareness: especially tricky for me around hit rate limits, token caps, text doubling bugs, deprecated packages, and truncation so many stages
- Prompt comparison scores Chain-of-Thought lowest -> not because it is a bad technique, but because it's the most token-hungry and therefore the most vulnerable to truncation



---



### Setup

In [1]:
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown
import time

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
MODEL = 'gemini-2.5-flash'

print("Gemini ready.")

Gemini ready.


### Load Earnings Call Transcript

In [2]:
BASE_QUESTION = (
    "Is Coinbase's strategy to reduce reliance on volatile trading fees "
    "actually working as of Q3 2024?"
)

# ── Condensed transcript — used in all four prompts for speed ─────────────────
TRANSCRIPT_SHORT = """
SOURCE: Coinbase Q3 2024 Earnings Call — October 30, 2024

BRIAN ARMSTRONG (CEO):
- Seventh consecutive quarter of positive adjusted EBITDA.
- Fourth consecutive quarter of positive net income.
- Subscription & services revenue on pace to exceed $2B in 2024, up from $1.4B in 2023.
- Strategy explicitly focused on diversifying AWAY from volatile transaction fees.
- USDC market cap up 45% YTD to $36B (from $25B); fastest-growing major USD stablecoin.
- Base (Layer 2) is #1 by transactions processed and total value locked.
  Transactions on Base up 55% Q/Q while median fee kept below $0.01.
- Stablecoin payments volume: $10T in 2023 already grew to $20T in 2024.

ALESIA HAAS (CFO):
- Q3 total revenue: $1.2B | Adjusted EBITDA: $449M | Net income: $75M.
- Total trading volume: $185B, DOWN 18% Q/Q.
- Transaction revenue: $573M, DOWN 27% Q/Q (lower volatility + lower asset prices).
- Subscription & services revenue: $556M, DOWN 7% Q/Q.
  (Native unit growth in staking/custody offset by lower crypto asset prices.)
- Stablecoin revenue UP 3% Q/Q — USDC growth exceeded headwind from lower interest rates.
- USD resources: $8.2B, up 5% Q/Q. First-ever $1B stock buyback authorised.
- Three financial goals: (1) diversify revenue, (2) expense discipline,
  (3) positive EBITDA in all market conditions.

Q&A KEY POINTS:
- Retail blended fee rate fell Q/Q due to MIX SHIFT toward stablecoin pair trades
  (which generate little-to-no fees). No changes to underlying fee structure.
- CFO: stablecoin mix shift is 'not guaranteed to persist' — not confirmed structural.
- Fiat-to-crypto market share was STEADY Q/Q — no market share loss.
- Derivatives revenue is beginning to grow but described as 'not yet material'.
- MiFID licence acquired (Aug 2024) to unlock derivatives in 20+ EU markets.
"""

# ── Full transcript — used ONLY in the executive summary for fact-checking ────
TRANSCRIPT_FULL = """
SOURCE: Coinbase Global (COIN) Q3 2024 Earnings Call — October 30, 2024

BRIAN ARMSTRONG (CEO):
We had some softer market conditions in Q3, but overall, it was a really solid quarter
for Coinbase. Seventh consecutive quarter of positive adjusted EBITDA. Fourth consecutive
quarter of positive net income. We've made a big effort to diversify our revenue away
from transaction fee revenue, which is more volatile, not as predictable, and more market
dependent. We've shifted more of that to subscription and services revenue over time.
We're now on pace to surpass $2 billion in subscription and services revenue in 2024,
up from $1.4 billion in 2023. The market cap of USDC is up 45% YTD to $36 billion,
up from $25 billion at the start of this year. Base is now the #1 Layer 2 solution by
transactions processed and total value on the platform. Transactions on Base increased
55% Q/Q while median fee kept below $0.01.

ALESIA HAAS (CFO):
Q3 total revenue: $1.2B. Adjusted EBITDA: $449M. Net income: $75M.
Total trading volume: $185B, down 18% Q/Q. Transaction revenue: $573M, down 27% Q/Q.
Subscription and services revenue: $556M, down 7% Q/Q. Native unit growth in staking
and custody offset by lower average crypto asset prices. Stablecoin revenue grew 3% Q/Q.
Subscription and services on pace to exceed $2B in 2024 vs $1.4B in 2023.
Three financial goals: (1) diversify revenue, (2) expense discipline,
(3) positive adjusted EBITDA in all market conditions. $1B buyback authorised.

Q&A:
- Retail blended fee rate declined due to mix shift toward stablecoin pair trades
  (little-to-no fees). No material changes to fee rate structure.
- CFO on structural nature: 'We see different mixes every quarter. Not guaranteed
  to persist.' Stablecoin impact was the most material contributor to rate change.
- Fiat-to-crypto U.S. market share steady Q/Q.
- Derivatives revenue is beginning to grow but described as 'not yet material'.
- MiFID licence acquired Aug 2024 for 20+ EU markets.
- $8.2B USD resources, up 5% Q/Q.
"""

print(f"SHORT transcript : {len(TRANSCRIPT_SHORT):,} chars")
print(f"FULL  transcript : {len(TRANSCRIPT_FULL):,} chars")

SHORT transcript : 1,799 chars
FULL  transcript : 2,024 chars


### Helper Function

In [3]:
def ask_gemini(prompt: str, temperature: float = 0.3, max_tokens: int = 4096) -> str:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
        )
    )
    try:
        parts = response.candidates[0].content.parts
        return "".join(part.text for part in parts if hasattr(part, "text"))
    except Exception as e:
        print(f"Error extracting response: {e}")
        return "ERROR: Could not extract response."

### Prompting Technique 1 - Zero-Shot Prompting

No examples or special instructions are provided beyond the question and the context. The model draws solely on its own reasoning ability and the transcript text.

In [4]:
prompt_zero_shot = f"""
Read the following earnings call transcript excerpt and answer the question.

TRANSCRIPT:
{TRANSCRIPT_SHORT}

QUESTION: {BASE_QUESTION}
"""

response_zero_shot = ask_gemini(prompt_zero_shot, temperature=0.3)
print("=" * 60)
print("TECHNIQUE 1: ZERO-SHOT")
print("=" * 60)
display(Markdown(response_zero_shot))

time.sleep(30)

TECHNIQUE 1: ZERO-SHOT


Yes, Coinbase's strategy to reduce reliance on volatile trading fees appears to be working, though with some mixed results in Q3 2024 due to market conditions.

Here's the evidence:

1.  **Explicit Strategy:** Both the CEO and CFO explicitly state that diversifying revenue away from volatile transaction fees is a core strategic goal.
2.  **Revenue Mix Shift:**
    *   In Q3 2024, **Subscription & services revenue ($556M)** is nearly equal to **Transaction revenue ($573M)**. This close parity indicates a significant shift in the revenue mix, moving away from a dominant reliance on transaction fees.
    *   The CEO notes that Subscription & services revenue is "on pace to exceed $2B in 2024, up from $1.4B in 2023," showing strong annual growth in this diversified stream.
3.  **Growth in Non-Transaction Streams:**
    *   **Stablecoin revenue** was **UP 3% Q/Q**, driven by USDC growth, despite headwinds from lower interest rates.
    *   **Derivatives revenue** is "beginning to grow," and the acquisition of a MiFID licence in August 2024 positions them for future growth in 20+ EU markets, indicating future diversification.
4.  **Reduced Transaction Fee Reliance (even if revenue is down):**
    *   **Transaction revenue was DOWN 27% Q/Q** due to lower volatility and asset prices. While this impacts total revenue, it also means the company is less reliant on this specific volatile stream.
    *   The **retail blended fee rate fell Q/Q** due to a mix shift toward stablecoin pair trades, which generate little-to-no fees. This further illustrates a move away from high-fee, volatile transactions, even if it impacts overall fee rates.

**However, there are nuances:**

*   **Q/Q Dip in Subscription & Services:** While growing annually, Subscription & services revenue was **DOWN 7% Q/Q** in Q3, primarily due to lower crypto asset prices offsetting native unit growth in staking/custody. This shows that even diversified revenue streams can be influenced by market conditions.
*   **Stablecoin Mix Shift Caution:** The CFO noted that the stablecoin mix shift is "not guaranteed to persist" and is "not confirmed structural," adding a note of caution to this specific trend.

In summary, the near-parity of transaction and subscription & services revenue, coupled with explicit strategic focus and growth in stablecoin revenue, strongly suggests that Coinbase's strategy to reduce reliance on volatile trading fees is indeed working, despite some quarter-over-quarter fluctuations influenced by broader market dynamics.

### Prompting Technique 2 - Few-Shot Prompting

Before asking the real question, we provide one or two worked examples (question + ideal answer) that demonstrate the format, depth, and structure we want. The model learns by analogy from those examples.

In [5]:
prompt_few_shot = f"""
You are a financial analyst. Study the two examples below, then answer the real
question at the end using the same format: cite specific figures, identify both
supporting evidence and counterpoints, give a clear verdict.

---
EXAMPLE 1
Question: Is Coinbase successfully controlling operating expenses?
Answer: Yes, with clear evidence. Total Q3 operating expenses were $1.0 billion,
down 6% Q/Q, while adjusted EBITDA came in at $449 million — the seventh consecutive
quarter of positive adjusted EBITDA. The CFO reaffirmed commitment to expense discipline
with only selective headcount additions in growth areas. This demonstrates structural
cost control, not one-off savings.

---
EXAMPLE 2
Question: Is Coinbase's stablecoin strategy gaining traction?
Answer: Yes, meaningfully. USDC market cap grew 45% YTD to $36B in Q3, up from $25B,
making it the fastest-growing major USD-backed stablecoin. Stablecoin revenue grew 3%
Q/Q despite lower interest rates. However, the CFO noted the mix shift toward stablecoin
pair trades — which generate little-to-no fees — is not guaranteed to persist,
introducing uncertainty about the blended fee rate going forward.

---
Now answer the real question using the same format.

TRANSCRIPT:
{TRANSCRIPT_SHORT}

QUESTION: {BASE_QUESTION}
"""

response_few_shot = ask_gemini(prompt_few_shot, temperature=0.3)

print("=" * 60)
print("TECHNIQUE 2: FEW-SHOT")
print("=" * 60)
display(Markdown(response_few_shot))

time.sleep(30)

TECHNIQUE 2: FEW-SHOT


Yes, meaningfully, with clear progress in diversifying revenue streams.

**Supporting Evidence:**
*   CEO Brian Armstrong explicitly stated the strategy is "focused on diversifying AWAY from volatile transaction fees."
*   Subscription & services revenue is on pace to exceed $2 billion in 2024, a significant increase from $1.4 billion in 2023, demonstrating strong annual growth in a non-transaction segment.
*   In Q3 2024, Subscription & services revenue ($556 million) was nearly on par with Transaction revenue ($573 million), indicating a substantial shift in the revenue mix compared to historical reliance on trading fees.
*   Stablecoin revenue grew 3% Q/Q, with USDC market cap up 45% YTD to $36 billion, showing growth and resilience in this non-transaction stream despite lower interest rates.
*   The Base Layer 2 network is growing rapidly, with transactions up 55% Q/Q, representing a new ecosystem for future revenue diversification.
*   CFO Alesia Haas reaffirmed "diversify revenue" as one of the company's three core financial goals.

**Counterpoints:**
*   Transaction revenue remains the largest single component at $573 million in Q3 and is still highly volatile, having decreased 27% Q/Q due to lower volatility and asset prices.
*   While growing annually, Subscription & services revenue actually declined 7% Q/Q in Q3, indicating it is not entirely immune to broader crypto market conditions, even if underlying unit growth is present.
*   The retail blended fee rate fell Q/Q due to a mix shift toward stablecoin pair trades, which generate little-to-no fees, introducing a potential headwind for future transaction revenue.
*   The CFO noted that this stablecoin mix shift is "not guaranteed to persist," suggesting it's not yet a confirmed structural change that would permanently reduce reliance on higher-fee trades.
*   Derivatives revenue, while a diversification effort, is described as "not yet material."

**Conclusion:** Coinbase's strategy to reduce reliance on volatile trading fees is demonstrably working, as evidenced by the significant growth and increasing proportion of non-transaction revenues like Subscription & Services and Stablecoin revenue. While transaction fees remain volatile and a major component, the company is making clear progress in rebalancing its revenue mix, though challenges like the Q/Q decline in S&S and the uncertain nature of stablecoin trade mix shifts highlight that the transition is ongoing.

### Prompting Method 3 - Chain of Thought

We explicitly instruct the model to reason step-by-step before reaching a conclusion. The model is told to break the problem into sub-questions, analyse each one, and synthesise a final answer.

In [10]:
# First call — Steps 1 and 2 only
prompt_cot_part1 = f"""
You are a financial analyst. Answer Steps 1 and 2 only.

Step 1 — STRATEGY: What specific actions has Coinbase taken to reduce reliance
on transaction fee revenue?

Step 2 — REVENUE MIX: What do the Q3 2024 numbers say about the split between
transaction revenue and subscription & services revenue? How has it changed YoY?

TRANSCRIPT:
{TRANSCRIPT_SHORT}
"""

response_cot_part1 = ask_gemini(prompt_cot_part1, temperature=0.3, max_tokens=1000)
time.sleep(30)

# Second call — Steps 3 and 4
prompt_cot_part2 = f"""
You are a financial analyst. Answer Steps 3 and 4 only.

Step 3 — HEADWINDS: What evidence suggests the strategy is incomplete or facing risks?

Step 4 — VERDICT: Synthesise into a direct answer with a confidence level
(strongly yes / partially / no).

TRANSCRIPT:
{TRANSCRIPT_SHORT}

CONTEXT FROM STEPS 1-2:
{response_cot_part1}
"""

response_cot_part2 = ask_gemini(prompt_cot_part2, temperature=0.3, max_tokens=1000)

# Combine both parts
response_cot = response_cot_part1 + "\n\n" + response_cot_part2

print("=" * 60)
print("TECHNIQUE 3: CHAIN-OF-THOUGHT (CoT)")
print("=" * 60)
display(Markdown(response_cot))

time.sleep(30)

TECHNIQUE 3: CHAIN-OF-THOUGHT (CoT)


As a financial analyst, here are the answers to your steps:

---

**Step 1 — STRATEGY: What specific actions has Coinbase taken to reduce reliance on transaction fee revenue?

**Step 3 — HEADWINDS:**

The evidence suggests that while Coinbase's diversification strategy is in motion and showing some positive signs, it is incomplete and facing several risks:

-> truncated / super short generation due to CoT being very token hungry

### Prompting Technique 4 - Role-Based Prompting

We assign the model a specific professional persona. The persona shapes the tone, vocabulary, analytical frame, and what counts as a satisfying answer.

In [7]:
prompt_role_based = f"""
You are a sceptical senior equity research analyst at a top-tier investment bank.
You are writing a section of a client research note for institutional investors
deciding whether to hold or trim their COIN position.

Your job is to cut through management's narrative and give a rigorous,
evidence-based assessment. Cite specific numbers. Be willing to say the data
does not yet fully support management's claims if that is what you find.
Highlight both progress and risks in equal measure.

Write 3 short paragraphs under the heading:
"Revenue Diversification: Progress Real, But Still Incomplete"

TRANSCRIPT:
{TRANSCRIPT_SHORT}

QUESTION TO ADDRESS: {BASE_QUESTION}
"""

response_role_based = ask_gemini(prompt_role_based, temperature=0.3, max_tokens=2048)

print("=" * 60)
print("TECHNIQUE 4: ROLE-BASED")
print("=" * 60)
display(Markdown(response_role_based))

time.sleep(30)

TECHNIQUE 4: ROLE-BASED


### Revenue Diversification: Progress Real, But Still Incomplete

Coinbase's strategic pivot towards diversifying revenue away from volatile transaction fees shows tangible progress, particularly in scaling its Subscription & Services segment. Management projects this segment to exceed $2 billion in 2024, a significant increase from $1.4 billion in 2023, demonstrating a clear growth trajectory for these non-transactional streams. Furthermore, the 45% YTD growth in USDC market cap to $36 billion and the robust expansion of Base, now #1 by transactions and TVL with 55% Q/Q transaction growth, lay foundational elements for future revenue generation, even if their direct contribution isn't yet fully realized. Stablecoin revenue itself was up 3% Q/Q, proving resilient against lower interest rates.

However, a rigorous look at Q3 2024 data reveals that the goal of reducing *reliance* on transaction fees remains incomplete. Transaction revenue, at $573 million, still marginally outpaced Subscription & Services revenue, which stood at $556 million. Critically, both segments experienced sequential declines: transaction revenue dropped 27% Q/Q, while Subscription & Services revenue fell 7% Q/Q, indicating that even diversified streams are not entirely immune to broader crypto market conditions (attributed to lower crypto asset prices). The Q&A further highlighted a concerning mix shift towards stablecoin pair trades, which generate "little-to-no fees," directly impacting the retail blended fee rate and prompting the CFO to caution that this shift is "not guaranteed to persist."

Ultimately, while the direction of travel is correct, Coinbase is not yet insulated from the inherent volatility of the crypto market. The Q3 figures underscore that despite growth in new areas, transaction fees remain a dominant, albeit shrinking, component of the revenue mix. The nascent derivatives business is "not yet material," and even the diversified Subscription & Services revenue proved susceptible to asset price fluctuations. Investors should view Coinbase's diversification as an ongoing, multi-year effort, with Q3 data confirming real progress in building new revenue pillars, but also highlighting the persistent challenges and the distance yet to cover before achieving true independence from market cycles.

### LLM as Judge

**Evaluation rubric:**

| Criterion | Description |
|-----------|-------------|
| Accuracy | Does it correctly represent the transcript data? |
| Depth | Does it go beyond surface-level description? |
| Evidence | Are specific figures from the transcript cited? |
| Balance | Does it acknowledge both positives and limitations? |
| Actionability | Does the conclusion help a reader make a decision? |

In [11]:
time.sleep(30)

judge_prompt = f"""
You are an expert evaluator assessing the quality of four AI-generated responses to
the same financial analysis question.

BASE QUESTION: {BASE_QUESTION}

TRANSCRIPT CONTEXT (used by all four responses):
{TRANSCRIPT_FULL}

---
RESPONSE A (Zero-Shot Prompt):
{response_zero_shot[:800]}

---
RESPONSE B (Few-Shot Prompt):
{response_few_shot[:800]}

---
RESPONSE C (Chain-of-Thought Prompt):
{response_cot[:800]}

---
RESPONSE D (Role-Based Prompt):
{response_role_based[:800]}

---
EVALUATION TASK:
Score each response from 1-5 on five criteria:
1. Accuracy       — Does it correctly represent the transcript data?
2. Depth          — Does it go beyond surface-level description?
3. Evidence       — Are specific figures from the transcript cited?
4. Balance        — Does it acknowledge both positives and limitations?
5. Actionability  — Does the conclusion help a decision-maker act?

FORMAT your response as follows:

### Scores
| Criterion     | Response A | Response B | Response C | Response D |
|---------------|------------|------------|------------|------------|
| Accuracy      | /5         | /5         | /5         | /5         |
| Depth         | /5         | /5         | /5         | /5         |
| Evidence      | /5         | /5         | /5         | /5         |
| Balance       | /5         | /5         | /5         | /5         |
| Actionability | /5         | /5         | /5         | /5         |
| **Total**     | /25        | /25        | /25        | /25        |

### Commentary
For each response, write 2-3 sentences explaining the scores, citing specific strengths
and weaknesses.

### Best Response
State which response is best and why in 2-3 sentences.

### Key Insight for Executive Summary
Based on the best elements across all four responses, identify the 3 most important
facts or arguments that MUST appear in a 200-word executive summary answering the
base question.
"""

judge_response = ask_gemini(judge_prompt, temperature=0.2, max_tokens=4096)

print("=" * 60)
print("LLM-AS-JUDGE EVALUATION")
print("=" * 60)
display(Markdown(judge_response))

LLM-AS-JUDGE EVALUATION


### Scores
| Criterion     | Response A | Response B | Response C | Response D |
|---------------|------------|------------|------------|------------|
| Accuracy      | 4/5        | 4/5        | 1/5        | 4/5        |
| Depth         | 2/5        | 2/5        | 1/5        | 3/5        |
| Evidence      | 4/5        | 4/5        | 1/5        | 4/5        |
| Balance       | 3/5        | 2/5        | 1/5        | 4/5        |
| Actionability | 1/5        | 1/5        | 1/5        | 2/5        |
|

### Executive Summary

In [21]:
time.sleep(30)

summary_prompt = f"""
You are a financial writer preparing a briefing for a senior executive.

Write a polished, balanced Executive Summary of exactly 200 words answering:

QUESTION: {BASE_QUESTION}

BASE YOUR ANSWER ON THIS TRANSCRIPT:
{TRANSCRIPT_SHORT}

REQUIREMENTS:
- First sentence: one clear verdict (yes, partially, or no).
- Cite at least three specific figures from the transcript.
- Acknowledge the single most important caveat or risk.
- Final sentence: a forward-looking implication for investors.
- Tone: professional, concise, balanced — suitable for a C-suite briefing.
- Length: exactly 200 words. Count carefully before responding.
"""

executive_summary = ask_gemini(summary_prompt, temperature=0.3, max_tokens=1500)
print("=" * 60)
print("FINAL EXECUTIVE SUMMARY (200 words)")
print("=" * 60)
display(Markdown(executive_summary))

word_count = len(executive_summary.split())
print(f"\nWord count: {word_count} words")

response = client.models.generate_content(
    model=MODEL,
    contents=summary_prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=1500,
    )
)

# Print full diagnostic
print(f"Finish reason: {response.candidates[0].finish_reason}")
print(f"Parts count: {len(response.candidates[0].content.parts)}")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"Part {i} length: {len(part.text)} chars")

executive_summary = "".join(part.text for part in response.candidates[0].content.parts)
print(f"\nTotal chars: {len(executive_summary)}")
print(f"Word count: {len(executive_summary.split())}")
display(Markdown(executive_summary))


FINAL EXECUTIVE SUMMARY (200 words)


Coinbase's strategy to reduce reliance on volatile trading fees is partially working, demonstrating progress in diversification despite persistent market headwinds. The company reported its seventh consecutive quarter of positive adjusted EBITDA and fourth of positive net income, indicating operational resilience. Subscription & services revenue is on pace to exceed $2 billion


Word count: 51 words
Finish reason: FinishReason.MAX_TOKENS
Parts count: 1
Part 0 length: 575 chars

Total chars: 575
Word count: 85


Coinbase's strategy to reduce reliance on volatile trading fees is partially working as of Q3 2024, showing progress in diversification despite immediate headwinds. The company reported its seventh consecutive quarter of positive adjusted EBITDA and fourth of positive net income, demonstrating operational resilience. Subscription & services revenue is on pace to exceed $2 billion in 2024, a significant increase from $1.4 billion in 2023, highlighting growth in non-transaction streams like staking and custody. USDC market cap grew 45% YTD to $36 billion, and Base (Layer